# ML Model Comparison Notebook
## Part 1.2 — Automated Trading System
### Comparing 6 Classification Models across 5 Stock Tickers

**Tickers:** AAPL, AMZN, GOOG, MSFT, TSLA  
**Models tested:** Logistic Regression, Random Forest, Gradient Boosting, AdaBoost, SVM, KNN  
**Task:** Binary classification — predict if price goes UP (1) or DOWN (0) next day  
**Train/Test split:** 80/20, no shuffle (time-series rule: train on past, test on future)  

---

## 1. Setup — Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from etl import FEATURE_COLUMNS

TICKERS = ['AAPL', 'AMZN', 'GOOG', 'MSFT', 'TSLA']
PROCESSED_DIR = os.path.join('..', 'data', 'processed')

print('All imports successful.')
print(f'Features used: {FEATURE_COLUMNS}')

## 2. Load and Explore the Processed Data

Data was prepared by Person 1's ETL pipeline (`src/etl.py`).  
Each CSV has one row per trading day with 8 engineered features and a binary target (1=UP, 0=DOWN).

In [ ]:
# Preview AAPL data
df_sample = pd.read_csv(os.path.join(PROCESSED_DIR, 'AAPL_processed.csv'))
print(f'Shape: {df_sample.shape}')
print(f'\nColumns: {df_sample.columns.tolist()}')
print(f'\nTarget distribution:')
print(df_sample['Target'].value_counts().rename({0: 'DOWN (0)', 1: 'UP (1)'}))
print(f'\nSample rows:')
df_sample[FEATURE_COLUMNS + ['Target']].head()

In [ ]:
# Class balance across all tickers
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ticker in enumerate(TICKERS):
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f'{ticker}_processed.csv'))
    counts = df['Target'].value_counts()
    axes[i].bar(['DOWN', 'UP'], [counts.get(0, 0), counts.get(1, 0)],
                color=['#e74c3c', '#2ecc71'])
    axes[i].set_title(ticker)
    axes[i].set_ylabel('Count')
plt.suptitle('Class Distribution per Ticker (DOWN=0, UP=1)', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Define All Models

All models share the same setup:
- Same `FEATURE_COLUMNS` from `etl.py`
- Same 80/20 train/test split with `shuffle=False`
- `StandardScaler` fitted **only on training data** (no data leakage)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced'
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, random_state=42
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=200, random_state=42
    ),
    'SVM': SVC(
        kernel='rbf', random_state=42, class_weight='balanced', probability=True
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=10
    ),
}

print('Models defined:')
for name in models:
    print(f'  {name}')

---
## 4. Model 1: Logistic Regression (Current Baseline)

**What it is:** A linear classifier that estimates the probability of UP or DOWN using a sigmoid function.

**Why use it:** Simple, fast, and fully interpretable. This is the model currently used by the team — we use it as the baseline to beat.

**Key setting:** `class_weight='balanced'` compensates for unequal UP/DOWN days in the data.

In [ ]:
model_name = 'Logistic Regression'
lr_accs = []

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
fig.suptitle(f'{model_name} — Confusion Matrix per Ticker', fontsize=13)

for i, ticker in enumerate(TICKERS):
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f'{ticker}_processed.csv'))
    X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    y = df['Target']
    mask = X.notna().all(axis=1)
    X, y = X[mask], y[mask]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    clf.fit(scaler.fit_transform(X_train), y_train)
    y_pred = clf.predict(scaler.transform(X_test))
    acc = accuracy_score(y_test, y_pred)
    lr_accs.append(acc)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues', cbar=False,
                xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'])
    axes[i].set_title(f'{ticker}\n{acc:.1%}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print(f'\n{model_name} — Results')
print('-' * 35)
for ticker, acc in zip(TICKERS, lr_accs):
    print(f'  {ticker}: {acc:.1%}')
print(f'  AVERAGE: {sum(lr_accs)/len(lr_accs):.1%}')

---
## 5. Model 2: Random Forest

**What it is:** An ensemble of 200 decision trees. Each tree is trained on a random sample of rows and features. The final prediction is a majority vote across all trees.

**Why use it:** Handles non-linear patterns that Logistic Regression misses. Naturally resistant to overfitting. Also gives us **feature importances** — we can see which indicators matter most.

**Key setting:** `n_estimators=200` — 200 trees gives a stable result without being too slow.

In [ ]:
model_name = 'Random Forest'
rf_accs = []

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
fig.suptitle(f'{model_name} — Confusion Matrix per Ticker', fontsize=13)

for i, ticker in enumerate(TICKERS):
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f'{ticker}_processed.csv'))
    X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    y = df['Target']
    mask = X.notna().all(axis=1)
    X, y = X[mask], y[mask]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
    clf.fit(scaler.fit_transform(X_train), y_train)
    y_pred = clf.predict(scaler.transform(X_test))
    acc = accuracy_score(y_test, y_pred)
    rf_accs.append(acc)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Greens', cbar=False,
                xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'])
    axes[i].set_title(f'{ticker}\n{acc:.1%}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print(f'\n{model_name} — Results')
print('-' * 35)
for ticker, acc in zip(TICKERS, rf_accs):
    print(f'  {ticker}: {acc:.1%}')
print(f'  AVERAGE: {sum(rf_accs)/len(rf_accs):.1%}')

In [ ]:
# Feature importances from Random Forest (AAPL example)
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'AAPL_processed.csv'))
X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
y = df['Target']
mask = X.notna().all(axis=1)
X, y = X[mask], y[mask]
X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, shuffle=False)
scaler = StandardScaler()
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(scaler.fit_transform(X_train), y_train)

importances = pd.Series(rf.feature_importances_, index=FEATURE_COLUMNS).sort_values()
plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color='#2ecc71')
plt.title('Random Forest — Feature Importances (AAPL)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Feature ranking (most important first):')
for feat, imp in importances.sort_values(ascending=False).items():
    print(f'  {feat:<20}: {imp:.4f}')

---
## 6. Model 3: Gradient Boosting

**What it is:** Builds trees one by one — each new tree focuses on correcting the mistakes made by the previous ones. Learns slowly but precisely.

**Why use it:** Often the strongest performer on tabular data. More powerful than Random Forest but takes longer to train.

**Key setting:** `n_estimators=200` — 200 sequential boosting rounds.

In [ ]:
model_name = 'Gradient Boosting'
gb_accs = []

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
fig.suptitle(f'{model_name} — Confusion Matrix per Ticker', fontsize=13)

for i, ticker in enumerate(TICKERS):
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f'{ticker}_processed.csv'))
    X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    y = df['Target']
    mask = X.notna().all(axis=1)
    X, y = X[mask], y[mask]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    clf = GradientBoostingClassifier(n_estimators=200, random_state=42)
    clf.fit(scaler.fit_transform(X_train), y_train)
    y_pred = clf.predict(scaler.transform(X_test))
    acc = accuracy_score(y_test, y_pred)
    gb_accs.append(acc)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Oranges', cbar=False,
                xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'])
    axes[i].set_title(f'{ticker}\n{acc:.1%}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print(f'\n{model_name} — Results')
print('-' * 35)
for ticker, acc in zip(TICKERS, gb_accs):
    print(f'  {ticker}: {acc:.1%}')
print(f'  AVERAGE: {sum(gb_accs)/len(gb_accs):.1%}')

---
## 7. Model 4: AdaBoost ⭐ Best Overall

**What it is:** Adaptive Boosting — trains simple classifiers sequentially, putting more weight on the samples that were misclassified previously. Each round it adapts to the hard cases.

**Why use it:** Proved to be the **best overall model** in our tests (50.7% average). Works particularly well on MSFT and TSLA.

**Key setting:** `n_estimators=200` — 200 weak learners combined into one strong classifier.

In [ ]:
model_name = 'AdaBoost'
ada_accs = []

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
fig.suptitle(f'{model_name} — Confusion Matrix per Ticker', fontsize=13)

for i, ticker in enumerate(TICKERS):
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f'{ticker}_processed.csv'))
    X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    y = df['Target']
    mask = X.notna().all(axis=1)
    X, y = X[mask], y[mask]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    clf = AdaBoostClassifier(n_estimators=200, random_state=42)
    clf.fit(scaler.fit_transform(X_train), y_train)
    y_pred = clf.predict(scaler.transform(X_test))
    acc = accuracy_score(y_test, y_pred)
    ada_accs.append(acc)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Purples', cbar=False,
                xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'])
    axes[i].set_title(f'{ticker}\n{acc:.1%}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print(f'\n{model_name} — Results')
print('-' * 35)
for ticker, acc in zip(TICKERS, ada_accs):
    print(f'  {ticker}: {acc:.1%}')
print(f'  AVERAGE: {sum(ada_accs)/len(ada_accs):.1%}')

---
## 8. Model 5: SVM (Support Vector Machine)

**What it is:** Finds the optimal boundary (hyperplane) that best separates UP and DOWN trading days in feature space. The RBF kernel lets it draw non-linear boundaries.

**Why use it:** Effective when the number of features is relatively small compared to samples. Robust to outliers.

**Key setting:** `kernel='rbf'` — Radial Basis Function allows curved decision boundaries.

In [ ]:
model_name = 'SVM'
svm_accs = []

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
fig.suptitle(f'{model_name} — Confusion Matrix per Ticker', fontsize=13)

for i, ticker in enumerate(TICKERS):
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f'{ticker}_processed.csv'))
    X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    y = df['Target']
    mask = X.notna().all(axis=1)
    X, y = X[mask], y[mask]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    clf = SVC(kernel='rbf', random_state=42, class_weight='balanced', probability=True)
    clf.fit(scaler.fit_transform(X_train), y_train)
    y_pred = clf.predict(scaler.transform(X_test))
    acc = accuracy_score(y_test, y_pred)
    svm_accs.append(acc)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Reds', cbar=False,
                xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'])
    axes[i].set_title(f'{ticker}\n{acc:.1%}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print(f'\n{model_name} — Results')
print('-' * 35)
for ticker, acc in zip(TICKERS, svm_accs):
    print(f'  {ticker}: {acc:.1%}')
print(f'  AVERAGE: {sum(svm_accs)/len(svm_accs):.1%}')

---
## 9. Model 6: KNN (K-Nearest Neighbors)

**What it is:** No training phase — to predict a new day, it finds the 10 most similar past trading days (by feature distance) and takes a majority vote on what happened next.

**Why use it:** Simple intuition — if today's market looks like certain past days, tomorrow will likely behave similarly. Proved **best for AAPL** in our tests.

**Key setting:** `n_neighbors=10` — look at the 10 most similar historical trading days.

In [ ]:
model_name = 'KNN'
knn_accs = []

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
fig.suptitle(f'{model_name} — Confusion Matrix per Ticker', fontsize=13)

for i, ticker in enumerate(TICKERS):
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f'{ticker}_processed.csv'))
    X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    y = df['Target']
    mask = X.notna().all(axis=1)
    X, y = X[mask], y[mask]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    clf = KNeighborsClassifier(n_neighbors=10)
    clf.fit(scaler.fit_transform(X_train), y_train)
    y_pred = clf.predict(scaler.transform(X_test))
    acc = accuracy_score(y_test, y_pred)
    knn_accs.append(acc)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='YlGn', cbar=False,
                xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'])
    axes[i].set_title(f'{ticker}\n{acc:.1%}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print(f'\n{model_name} — Results')
print('-' * 35)
for ticker, acc in zip(TICKERS, knn_accs):
    print(f'  {ticker}: {acc:.1%}')
print(f'  AVERAGE: {sum(knn_accs)/len(knn_accs):.1%}')

---
## 10. Final Summary — All Models Compared

In [ ]:
# Collect all results into a clean summary table
all_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest':        RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=200, random_state=42),
    'AdaBoost':             AdaBoostClassifier(n_estimators=200, random_state=42),
    'SVM':                  SVC(kernel='rbf', random_state=42, class_weight='balanced', probability=True),
    'KNN':                  KNeighborsClassifier(n_neighbors=10),
}

summary = {}
for name, clf in all_models.items():
    accs = []
    for ticker in TICKERS:
        df = pd.read_csv(os.path.join(PROCESSED_DIR, f'{ticker}_processed.csv'))
        X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
        y = df['Target']
        mask = X.notna().all(axis=1)
        X, y = X[mask], y[mask]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
        scaler = StandardScaler()
        clf.fit(scaler.fit_transform(X_train), y_train)
        accs.append(accuracy_score(y_test, clf.predict(scaler.transform(X_test))))
    summary[name] = accs + [sum(accs) / len(accs)]

df_summary = pd.DataFrame(summary, index=TICKERS + ['AVERAGE']).T
df_summary_pct = df_summary.applymap(lambda x: f'{x:.1%}')
print('Accuracy Summary Table')
print('=' * 65)
print(df_summary_pct.to_string())

In [ ]:
# Heatmap visualisation
plt.figure(figsize=(11, 5))
sns.heatmap(
    df_summary,
    annot=df_summary_pct,
    fmt='',
    cmap='RdYlGn',
    vmin=0.42, vmax=0.58,
    linewidths=0.5,
    annot_kws={'size': 11}
)
plt.title('Accuracy Heatmap — All Models vs All Tickers', fontsize=14, pad=15)
plt.xlabel('Ticker / Average', fontsize=11)
plt.ylabel('Model', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparing average accuracy per model
averages = {name: vals[-1] for name, vals in summary.items()}
colors = ['#3498db' if name == 'Logistic Regression' else '#e74c3c' if v == max(averages.values()) else '#95a5a6'
          for name, v in averages.items()]

plt.figure(figsize=(10, 4))
bars = plt.bar(averages.keys(), averages.values(), color=colors, edgecolor='white', linewidth=0.8)
plt.axhline(y=0.5, color='black', linestyle='--', alpha=0.4, label='50% baseline (random)')
for bar, val in zip(bars, averages.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.1%}', ha='center', va='bottom', fontsize=10)
plt.ylim(0.44, 0.54)
plt.title('Average Accuracy per Model (blue = current baseline, red = best)', fontsize=12)
plt.ylabel('Accuracy')
plt.xticks(rotation=15)
plt.legend()
plt.tight_layout()
plt.show()

best_model = max(averages, key=averages.get)
print(f'\nCurrent baseline (Logistic Regression): {averages["Logistic Regression"]:.1%}')
print(f'Best model overall: {best_model} → {averages[best_model]:.1%}')
print(f'Improvement: +{(averages[best_model] - averages["Logistic Regression"]):.1%}')

In [ ]:
# Best model per ticker
print('Best model per ticker:')
print('=' * 45)
for ticker in TICKERS:
    best = max(summary, key=lambda n: summary[n][TICKERS.index(ticker)])
    best_acc = summary[best][TICKERS.index(ticker)]
    current  = summary['Logistic Regression'][TICKERS.index(ticker)]
    delta    = best_acc - current
    print(f'  {ticker}: {best:<22} {best_acc:.1%}  (+{delta:.1%} vs current)')

print(f'\nMixed approach average: {sum(summary[max(summary, key=lambda n: summary[n][i])][i] for i, _ in enumerate(TICKERS)) / len(TICKERS):.1%}')

---
## 11. Conclusion

### Results Summary

| Model | AAPL | AMZN | GOOG | MSFT | TSLA | **AVG** |
|-------|------|------|------|------|------|---------|
| Logistic Regression (baseline) | 48.0% | 50.3% | 45.9% | 51.4% | 46.6% | 48.4% |
| Random Forest | 51.4% | 50.7% | 50.3% | 50.0% | 47.3% | 49.9% |
| Gradient Boosting | 43.6% | 45.9% | 46.3% | 52.4% | 49.7% | 47.6% |
| **AdaBoost** | 51.0% | 51.4% | 44.3% | 53.0% | 54.1% | **50.7%** |
| SVM | 48.6% | 46.3% | 49.3% | 49.7% | 49.7% | 48.7% |
| KNN | **51.7%** | 50.0% | 49.0% | 53.0% | 47.6% | 50.3% |
| **Best per ticker** | KNN | AdaBoost | R.Forest | AdaBoost | AdaBoost | **52.1%** |

### Why ~50% accuracy is normal
Stock markets are highly efficient. Even professional quantitative funds rarely exceed 55% on daily direction prediction. The goal of this project is a **working ML pipeline**, not beating the market.

### Recommendation
- **AdaBoost** is the best single model (50.7% average, +2.3% over baseline)
- **Best-per-ticker mixed approach** gives the highest accuracy (52.1%, +3.7% over baseline)
- Both are clear, explainable improvements over the current Logistic Regression

---
## 12. Train Best Models and Save as .pkl Files

Based on the comparison above, we train the **best model per ticker** and save both the model and scaler as `.pkl` files into the `models/` folder.

| Ticker | Best Model | Accuracy |
|--------|-----------|----------|
| AAPL | KNN | 51.7% |
| AMZN | AdaBoost | 51.4% |
| GOOG | Random Forest | 50.3% |
| MSFT | AdaBoost | 53.0% |
| TSLA | AdaBoost | 54.1% |

Each ticker gets two files saved:
- `models/TICKER_model.pkl` — the trained classifier
- `models/TICKER_scaler.pkl` — the fitted StandardScaler

In [ ]:
import joblib

# Best model per ticker based on comparison results
BEST_MODELS = {
    'AAPL': KNeighborsClassifier(n_neighbors=10),
    'AMZN': AdaBoostClassifier(n_estimators=200, random_state=42),
    'GOOG': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
    'MSFT': AdaBoostClassifier(n_estimators=200, random_state=42),
    'TSLA': AdaBoostClassifier(n_estimators=200, random_state=42),
}

MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

print('Training and saving best models...\n')
print(f'{"Ticker":<8} {"Model":<25} {"Train rows":<12} {"Test rows":<12} {"Accuracy"}')
print('-' * 65)

for ticker, clf in BEST_MODELS.items():
    # Load processed data
    df = pd.read_csv(os.path.join(PROCESSED_DIR, f'{ticker}_processed.csv'))
    X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    y = df['Target']
    mask = X.notna().all(axis=1)
    X, y = X[mask], y[mask]

    # Train / test split — no shuffle (time-series rule)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )

    # Fit scaler on training data only
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # Train
    clf.fit(X_train_s, y_train)

    # Evaluate
    acc = accuracy_score(y_test, clf.predict(X_test_s))

    # Save model and scaler
    model_path  = os.path.join(MODELS_DIR, f'{ticker}_model.pkl')
    scaler_path = os.path.join(MODELS_DIR, f'{ticker}_scaler.pkl')
    joblib.dump(clf,    model_path)
    joblib.dump(scaler, scaler_path)

    model_name = clf.__class__.__name__
    print(f'{ticker:<8} {model_name:<25} {len(X_train):<12} {len(X_test):<12} {acc:.1%}')

print('\nAll models saved successfully!')

In [ ]:
# Verify all files were saved correctly
print('Verifying saved files...\n')
all_good = True
for ticker in BEST_MODELS:
    model_path  = os.path.join(MODELS_DIR, f'{ticker}_model.pkl')
    scaler_path = os.path.join(MODELS_DIR, f'{ticker}_scaler.pkl')

    m = joblib.load(model_path)
    s = joblib.load(scaler_path)

    m_ok = os.path.exists(model_path)
    s_ok = os.path.exists(scaler_path)
    all_good = all_good and m_ok and s_ok

    print(f'  {ticker}_model.pkl   → {m.__class__.__name__}  ✓')
    print(f'  {ticker}_scaler.pkl  → {s.__class__.__name__}  ✓')

print()
if all_good:
    print('All 10 .pkl files verified and ready.')
    print(f'Location: {os.path.abspath(MODELS_DIR)}')
else:
    print('WARNING: some files are missing — re-run the cell above.')